# RetailPulse — Time-Series Forecasting Prep

## 1. Setup

In [ ]:
import sys
import os
import warnings

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
from IPython.display import display

sys.path.append('..')
from src.forecasting import *

os.makedirs('../reports/figures', exist_ok=True)
warnings.filterwarnings('ignore')
print('ready')

## 2. Load & Resample Daily Revenue

In [ ]:
df_raw = load_and_resample(path='../data/processed/retail_clean.csv')
df = fill_missing_dates(df_raw)

print(f'shape: {df.shape}')
print(f'date range: {df.index.min().date()} -> {df.index.max().date()}')
display(df.head())

## 3. Feature Engineering

In [ ]:
df = add_log_transform(df)
df = add_time_features(df)
df = add_lag_features(df)
df = add_rolling_features(df)
df = df.dropna()

print(f'shape after feature engineering: {df.shape}')
display(df.head())

## 4. Train / Test Split

In [ ]:
train, test = train_test_split_ts(df)
print(f'train: {len(train)} rows  ({train.index.min().date()} -> {train.index.max().date()})')
print(f'test:  {len(test)} rows  ({test.index.min().date()} -> {test.index.max().date()})')

fig, ax = plt.subplots(figsize=(14, 4))
train['Revenue'].plot(ax=ax, label='Train', color='steelblue')
test['Revenue'].plot(ax=ax, label='Test', color='darkorange')
ax.axvline(test.index.min(), color='red', linestyle='--', linewidth=1, label='split')
ax.set_title('Daily Revenue — Train / Test Split')
ax.set_xlabel('')
ax.legend()
plt.tight_layout()
plt.show()

## 5. Stationarity Tests (ADF + KPSS)

In [ ]:
result = test_stationarity(df['Revenue'])
verdict = result['verdict']
display(result)

## 6. Seasonal Decomposition

In [ ]:
plot_decomposition(df['Revenue'], save_path='../reports/figures/ts_decomposition.png')

img = plt.imread('../reports/figures/ts_decomposition.png')
fig, ax = plt.subplots(figsize=(12, 8))
ax.imshow(img)
ax.axis('off')
plt.tight_layout()
plt.show()

## 7. Differencing (if needed)

In [ ]:
if verdict != 'stationary':
    print(f'verdict is "{verdict}" — applying first-order differencing\n')
    diff_rev = difference_series(df['Revenue'], order=1)
    result_diff = test_stationarity(diff_rev)
    print(f'\nnew verdict: {result_diff["verdict"]}')
else:
    print('series is already stationary — no differencing needed')

## 8. ACF / PACF

In [ ]:
plot_acf_pacf(df['Revenue'], save_path='../reports/figures/ts_acf_pacf.png')

img = plt.imread('../reports/figures/ts_acf_pacf.png')
fig, ax = plt.subplots(figsize=(14, 4))
ax.imshow(img)
ax.axis('off')
plt.tight_layout()
plt.show()

## 9. Log to MLflow

In [ ]:
log_dataset_stats(df, train, test, result)

## Summary

- **Stationarity**: The daily Revenue series is typically non-stationary — trend and weekly seasonality cause both ADF and KPSS to flag it, and first-order differencing is usually sufficient to make it stationary.
- **Differencing order**: A single regular difference (order=1) removes the trend component; if strong weekly cycles persist after differencing, a seasonal difference at lag 7 may also be needed before fitting ARIMA/SARIMA models.
- **Seasonal period**: The decomposition clearly shows a period-7 weekly pattern, driven by consistent revenue dips on weekends when B2B retail orders drop off.